# Pipeline walkthrough
Step-by-step tour of the `ctcancer` library. By default it uses the **synthetic software-test phantom** (geometric, not patient data); set `SCAN` to a real DICOM folder / NIfTI file once you have downloaded data.

In [ ]:
import sys; sys.path.insert(0, '../src')
import numpy as np, matplotlib.pyplot as plt
from ctcancer.config import load_config
from ctcancer.io import load_any
from ctcancer.phantom import make_phantom
cfg = load_config()
SCAN = None  # e.g. '../data/raw/NSCLC-Radiomics/LUNG1-001'
vol = load_any(SCAN) if SCAN else make_phantom()[0]
vol.summary()

## 1. Preprocessing — HU clipping, resampling, lung window

In [ ]:
from ctcancer.preprocessing import preprocess_volume
pre = preprocess_volume(vol, cfg['preprocessing'])
print(pre.steps)
z = pre.hu.shape[0] // 2
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(pre.hu[z], cmap='gray', vmin=-1350, vmax=150); ax[0].set_title('lung window (HU)')
ax[1].hist(pre.hu.ravel(), bins=100); ax[1].set_title('HU histogram'); plt.show()

## 2. Lung segmentation (classical baseline)

In [ ]:
from ctcancer.segmentation import classical_lung_mask, split_left_right, lung_volume_ml
lung = classical_lung_mask(pre.hu, pre.spacing)
lr = split_left_right(lung)
print('lung volume (mL):', round(lung_volume_ml(lung, pre.spacing)))
plt.imshow(pre.norm[z], cmap='gray'); plt.imshow(np.ma.masked_equal(lr[z], 0), alpha=0.35, cmap='coolwarm'); plt.show()

## 3. Candidate generation and lesion features

In [ ]:
from ctcancer.detection import generate_candidates, full_mask
from ctcancer.features import extract_radiomics, describe_location
cands = generate_candidates(pre.hu, lung, pre.spacing)
for c in cands[:5]:
    m = full_mask(c, pre.hu.shape)
    f = extract_radiomics(pre.hu, m, pre.spacing)
    print(c.id, describe_location(c.centroid_zyx, lr, lung)['text'], round(f['max_axial_diameter_mm'], 1), 'mm, sphericity', round(f['sphericity'], 2))

## 4. Full pipeline + model status
Without trained models every learned stage reports *Model not trained* — no probabilities are invented.

In [ ]:
from ctcancer.inference.pipeline import AnalysisPipeline
res, arrays = AnalysisPipeline(cfg).run(vol)
{k: v['badge'] for k, v in res.statuses.items()}

## 5. PCA vs LDA on real features
After `training/extract_features.py --dataset lidc`, load `data/features/lidc.csv` and use `ctcancer.classical.fit_projections` — or run `training/run_pca_lda.py`, which performs the full patient-grouped study.

In [ ]:
from ctcancer.classical import PCA_VS_LDA_TEXT
print(PCA_VS_LDA_TEXT)